In [1]:
# =============================================================================
# Celula 0: Fonte de dados
#
# Prioridade:
#   1. output/data/eda_latest_meta.json  — dados ja filtrados pelo 01_EDA
#   2. SQL (Oracle) ou CSV bruto         — fallback completo
# =============================================================================
import os
import sys
import json as _json0
from pathlib import Path

def _ensure_shared_path():
    cwd = Path.cwd()
    if (cwd / 'shared').exists():
        sys.path.insert(0, str(cwd))
    elif (cwd / 'notebooks' / 'shared').exists():
        sys.path.insert(0, str(cwd / 'notebooks'))

_ensure_shared_path()

DATA_DIR    = os.path.join('output', 'data')
METRICS_DIR = os.path.join('output', 'metrics')
os.makedirs(METRICS_DIR, exist_ok=True)
CONFIG_PATH = os.path.join(METRICS_DIR, 'data_source_config.json')

# --- 1. Tenta carregar meta do 01_EDA (dados ja filtrados) ---
_EDA_META_PATH = os.path.join(DATA_DIR, 'eda_latest_meta.json')
EDA_META = None

if os.path.exists(_EDA_META_PATH):
    with open(_EDA_META_PATH) as _mf:
        EDA_META = _json0.load(_mf)
    _filtered_csv = EDA_META.get('filtered_csv', '')
    if os.path.exists(_filtered_csv):
        DATA_SOURCE   = 'EDA_FILTERED'
        CSV_FILE      = _filtered_csv
        EDA_ID        = EDA_META.get('eda_id', 'unknown')
        COLLECTION_ID = EDA_META.get('collection_id', 'unknown')
        print('[AUTO] Dados filtrados do 01_EDA encontrados:')
        print(f'  EDA_ID:        {EDA_ID}')
        print(f'  collection_id: {COLLECTION_ID}')
        print(f'  Arquivo:       {_filtered_csv}')
        print(f'  Linhas:        {EDA_META.get("rows", "?")}')
        print(f'  Sample Hz:     {EDA_META.get("sample_hz", "?")}')
    else:
        print(f'[AVISO] eda_latest_meta.json encontrado mas CSV nao existe: {_filtered_csv}')
        print('[FALLBACK] Usando select_raw_source...')
        from shared.data_sources import select_raw_source
        DATA_SOURCE, CSV_FILE = select_raw_source(
            data_dir=DATA_DIR, config_path=CONFIG_PATH, allow_sql=True)
        EDA_ID = None; COLLECTION_ID = None
else:
    print('[INFO] eda_latest_meta.json nao encontrado. Usando select_raw_source...')
    from shared.data_sources import select_raw_source
    DATA_SOURCE, CSV_FILE = select_raw_source(
        data_dir=DATA_DIR, config_path=CONFIG_PATH, allow_sql=True)
    EDA_ID = None; COLLECTION_ID = None

print(f'\nDATA_SOURCE = {DATA_SOURCE!r}')


[AUTO] Dados filtrados do 01_EDA encontrados:
  EDA_ID:        eda_20260307_192958
  collection_id: col_20260307_185012_100hz
  Arquivo:       output\data\raw_sensor_data_filtered_20260307_193528.csv
  Linhas:        212095
  Sample Hz:     100.0

DATA_SOURCE = 'EDA_FILTERED'


In [2]:
# =============================================================================
# Celula 1: Configuracao e Imports
# =============================================================================
import os
import sys
import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats as scipy_stats
from scipy.signal import savgol_filter
from scipy.stats import skew, kurtosis, kruskal
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

# --- Bootstrap para importar modulos compartilhados (notebooks/shared) ---
def _ensure_shared_path():
    cwd = Path.cwd()
    if (cwd / 'shared').exists():
        sys.path.insert(0, str(cwd))
    elif (cwd / 'notebooks' / 'shared').exists():
        sys.path.insert(0, str(cwd / 'notebooks'))

_ensure_shared_path()

from shared.class_config import (
    CLASS_ORDER, COLOR_MAP, LEGACY_CLASS_ORDER, LEGACY_COLOR_MAP,
    derive_composite_label, get_active_classes, get_color_map,
    FILTER_LABELS_SQL,
)
from shared.data_sources import load_raw_from_oracle

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# --- CONFIGURACAO DO BANCO (ADB Cloud / XE local — via shared/db_oracle) ---
from shared.db_oracle import oracle_connect, get_engine, DB_MODE_STR, ORACLE_DSN_STR, ORACLE_USER, TABLE_TRAINING, TABLE_MONITORING, TABLE_LEGACY
print(f'[DB] modo: {DB_MODE_STR}')

# =============================================================================
# AUTO-DETECT: Ler parametros do sistema (control_state.json + banco Oracle)
# =============================================================================
CONTROL_STATE_PATH = os.path.join(
    '..', 'api', 'control_states', 'control_state_ESP32_MPU6050_ORACLE.json'
)
if not os.path.exists(CONTROL_STATE_PATH):
    CONTROL_STATE_PATH = os.path.join('..', 'api', 'control_state.json')

# 1. Ler control_state.json (fonte de verdade do servidor)
control_state = {}
if os.path.exists(CONTROL_STATE_PATH):
    with open(CONTROL_STATE_PATH, 'r') as f:
        control_state = json.load(f)
    print(f'[AUTO] control_state.json carregado: {CONTROL_STATE_PATH}')
    print(f'  Modo atual:      {control_state.get("mode", "N/A")}')
    print(f'  Sample rate:     {control_state.get("sample_rate", "N/A")} Hz')
    print(f'  Collection ID:   {control_state.get("collection_id", "N/A")}')
else:
    print(f'[AVISO] control_state.json nao encontrado em {CONTROL_STATE_PATH}')
    print(f'  Usando valores padrao.')

# 2. Ler taxa configurada (do control_state ou fallback)
SAMPLE_RATE_HZ = int(control_state.get('sample_rate', 20))
EFFECTIVE_SAMPLE_RATE_HZ = SAMPLE_RATE_HZ

# Sampling validation metadata (preenchido mais adiante; evita NameError na exportacao)
VALIDATION_METADATA = {
    'status': 'not_computed_yet',
    'expected_hz': float(EFFECTIVE_SAMPLE_RATE_HZ),
    'per_class': {},
}

# --- PARAMETROS DE FILTRO (opcionais) ---
FILTER_SAMPLE_RATE_HZ = None  # ex: 20 para filtrar; None = usar todas
FILTER_COLLECTION_ID = None   # ex: 'col_...' ou ['col_a', 'col_b']

# 7 classes compostas (derivadas de cmd_speed_label + rot_state_label)
FILTER_LABELS = CLASS_ORDER


# 3. Consultar Oracle para obter parametros reais da coleta
def _rows_to_dict(cursor, rows):
    """Converte resultado de cursor Oracle para lista de dicts."""
    cols = [d[0].lower() for d in cursor.description]
    return [dict(zip(cols, row)) for row in rows]

print(f'\n[AUTO] Consultando Oracle para validar parametros...')
try:
    _conn_auto = oracle_connect()
    _cursor = _conn_auto.cursor()

    # Total de amostras por cmd_speed_label
    _cursor.execute("""
        SELECT cmd_speed_label AS fan_state, COUNT(*) AS cnt,
               MIN(ts_epoch) AS ts_min, MAX(ts_epoch) AS ts_max,
               AVG(sample_rate) AS avg_sample_rate
        FROM sensor_training_data
        WHERE cmd_speed_label IN ('LOW', 'MEDIUM', 'HIGH', 'OFF')
        GROUP BY cmd_speed_label
        ORDER BY cnt DESC
    """)
    _class_stats = _rows_to_dict(_cursor, _cursor.fetchall())

    # Collections disponiveis
    _cursor.execute("""
        SELECT collection_id, COUNT(*) AS cnt,
               MIN(ts_epoch) AS ts_min, MAX(ts_epoch) AS ts_max
        FROM sensor_training_data
        WHERE cmd_speed_label IN ('LOW', 'MEDIUM', 'HIGH')
        GROUP BY collection_id
        ORDER BY MAX(ts_epoch) DESC
    """)
    _collection_stats = _rows_to_dict(_cursor, _cursor.fetchall())

    # Taxa real de sample_rate gravada no banco
    _cursor.execute("""
        SELECT ROUND(AVG(sample_rate), 2) AS db_avg_rate,
               ROUND(MIN(sample_rate), 2) AS db_min_rate,
               ROUND(MAX(sample_rate), 2) AS db_max_rate
        FROM sensor_training_data
        WHERE cmd_speed_label IN ('LOW', 'MEDIUM', 'HIGH')
    """)
    _rate_rows = _rows_to_dict(_cursor, _cursor.fetchall())
    _rate_stats = _rate_rows[0] if _rate_rows else {}

    _conn_auto.close()

    # Exibir resumo
    print(f'\n{"="*60}')
    print(f' PARAMETROS AUTO-DETECTADOS DO SISTEMA (Oracle)')
    print(f'{"="*60}')
    print(f'  SAMPLE_RATE_HZ (control_state): {SAMPLE_RATE_HZ} Hz')
    if _rate_stats and _rate_stats.get('db_avg_rate') is not None:
        print(f'  sample_rate no banco (media):   {_rate_stats["db_avg_rate"]} Hz')
        print(f'  sample_rate no banco (min/max):  {_rate_stats["db_min_rate"]} / {_rate_stats["db_max_rate"]} Hz')
        db_rate = float(_rate_stats['db_avg_rate'])
        if abs(db_rate - SAMPLE_RATE_HZ) / max(SAMPLE_RATE_HZ, 1) > 0.15:
            print(f'  *** ALERTA: Taxa no banco ({db_rate} Hz) difere do control_state ({SAMPLE_RATE_HZ} Hz)! ***')

    print(f'\n  Dados por cmd_speed_label no banco:')
    _total_training = 0
    for row in _class_stats:
        state = row['fan_state']
        count = int(row['cnt'])
        duration = (float(row['ts_max']) - float(row['ts_min'])) / 60 if row['ts_max'] and row['ts_min'] else 0
        marker = '  (inativo)' if state == 'OFF' else ''
        print(f'    {state:8s}: {count:6d} amostras, {duration:6.1f} min{marker}')
        if state in ('LOW', 'MEDIUM', 'HIGH'):
            _total_training += count
    print(f'    {"TOTAL":8s}: {_total_training:6d} amostras de treino')

    if _collection_stats:
        print(f'\n  Collections (treino):')
        for row in _collection_stats:
            duration = (float(row['ts_max']) - float(row['ts_min'])) / 60 if row['ts_max'] and row['ts_min'] else 0
            print(f'    {row["collection_id"]}: {int(row["cnt"])} amostras, {duration:.1f} min')

    print(f'{"="*60}')

except Exception as e:
    print(f'[AVISO] Nao foi possivel consultar Oracle: {e}')
    print(f'  Verifique se o servico Oracle esta rodando e as credenciais estao corretas.')
    print(f'  Usando SAMPLE_RATE_HZ = {SAMPLE_RATE_HZ} do control_state.')

# --- PARAMETROS DE FEATURE ENGINEERING ---
WINDOW_SIZE = 100    # pontos por janela
STEP_SIZE = 20       # stride da janela deslizante

# Feature selection
ANOVA_ALPHA = 0.05
CORRELATION_THRESHOLD = 0.85

# Eixos do sensor
SENSOR_AXES = ['accel_x_g', 'accel_y_g', 'accel_z_g', 'gyro_x_dps', 'gyro_y_dps', 'gyro_z_dps']

# Eixos/metricas candidatos do modelo
# Metricas estendidas: basicas + shape + percentis + FFT por banda
DERIVED_AXES = ['vibration_dps', 'accel_mag_g']
FEATURE_AXES = SENSOR_AXES + DERIVED_AXES
FEATURE_METRICS = [
    'std', 'range', 'rms',            # basicas (alinham com JS)
    'skew', 'kurtosis',               # shape
    'p10', 'p25', 'p75', 'p90', 'p95',  # percentis
    'fft_low', 'fft_mid', 'fft_high', # bandas FFT (0-5 Hz, 5-15 Hz, 15-10 Hz)
]
FEATURE_COLUMNS = [f'{ax}_{m}' for ax in FEATURE_AXES for m in FEATURE_METRICS]

# Suavizacao para graficos (apenas visual)
SMOOTH_WINDOW_S = 0.5
SMOOTH_POLYORDER = 3

# Paths
from shared.paths import get_paths
PATHS = get_paths()
TIMESTAMP_STR = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DATA = str(PATHS.output_data)
OUTPUT_FIGURES = str(PATHS.output_figures)
OUTPUT_METRICS = str(PATHS.output_metrics)
CONFIG_DIR = str(PATHS.config_dir)

def _savgol_smooth(series, sample_rate_hz, window_s=SMOOTH_WINDOW_S, polyorder=SMOOTH_POLYORDER):
    values = np.asarray(series, dtype=float)
    if values.size < 3:
        return values
    window_len = int(round(window_s * sample_rate_hz))
    if window_len < 3:
        window_len = 3
    if window_len % 2 == 0:
        window_len += 1
    if window_len > values.size:
        window_len = values.size if values.size % 2 == 1 else values.size - 1
    if window_len < 3:
        return values
    if polyorder >= window_len:
        polyorder = max(1, window_len - 1)
    try:
        return savgol_filter(values, window_length=window_len, polyorder=polyorder)
    except Exception:
        return values

print(f'\n--- Configuracao Final ---')
print(f'Timestamp:            {TIMESTAMP_STR}')
print(f'Oracle DSN:           {ORACLE_DSN_STR}')
print(f'Sample Rate:          {SAMPLE_RATE_HZ} Hz')
if FILTER_SAMPLE_RATE_HZ is not None:
    print(f'Filtro sample_rate:   {FILTER_SAMPLE_RATE_HZ} Hz')
if FILTER_COLLECTION_ID:
    print(f'Filtro collection_id: {FILTER_COLLECTION_ID}')
if FILTER_LABELS:
    print(f'Filtro labels:        {FILTER_LABELS}')
print(f'Window: {WINDOW_SIZE} pontos = {WINDOW_SIZE/SAMPLE_RATE_HZ:.0f}s, Step: {STEP_SIZE} pontos = {STEP_SIZE/SAMPLE_RATE_HZ:.0f}s')
print(f'Features candidatas por eixo: {len(FEATURE_METRICS)} metricas x {len(FEATURE_AXES)} eixos = {len(FEATURE_COLUMNS)} features')
print(f'Figuras:  {os.path.abspath(OUTPUT_FIGURES)}')
print(f'Metricas: {os.path.abspath(OUTPUT_METRICS)}')

# --- Override de SAMPLE_RATE_HZ se dados vierem do 01_EDA ---
if globals().get('EDA_META') and EDA_META is not None and EDA_META.get('sample_hz'):
    SAMPLE_RATE_HZ = int(float(EDA_META['sample_hz']))
    EFFECTIVE_SAMPLE_RATE_HZ = float(SAMPLE_RATE_HZ)
    VALIDATION_METADATA['expected_hz'] = EFFECTIVE_SAMPLE_RATE_HZ
    print(f'[EDA_FILTERED] SAMPLE_RATE_HZ = {SAMPLE_RATE_HZ} Hz (do 01_EDA meta)')
    # Ajusta janela: min(100, 5*Hz) pontos; step: max(20, Hz//5)
    WINDOW_SIZE = min(100, int(SAMPLE_RATE_HZ * 5))
    STEP_SIZE   = max(20, SAMPLE_RATE_HZ // 5)
    print(f'[EDA_FILTERED] WINDOW_SIZE={WINDOW_SIZE}  STEP_SIZE={STEP_SIZE}')


[DB] modo: ADB Cloud  DSN=mpuiotdb_tp
[AUTO] control_state.json carregado: ..\api\control_states\control_state_ESP32_MPU6050_ORACLE.json
  Modo atual:      PAUSE
  Sample rate:     100 Hz
  Collection ID:   col_20260307_185012_100hz

[AUTO] Consultando Oracle para validar parametros...

 PARAMETROS AUTO-DETECTADOS DO SISTEMA (Oracle)
  SAMPLE_RATE_HZ (control_state): 100 Hz
  sample_rate no banco (media):   100 Hz
  sample_rate no banco (min/max):  100 / 100 Hz

  Dados por cmd_speed_label no banco:
    HIGH    :  66664 amostras,   22.0 min
    LOW     :  66664 amostras,   22.0 min
    MEDIUM  :  66651 amostras,   22.0 min
    OFF     :  30342 amostras,    5.2 min  (inativo)
    TOTAL   : 199979 amostras de treino

  Collections (treino):
    col_20260307_185012_100hz: 199979 amostras, 33.0 min

--- Configuracao Final ---
Timestamp:            20260307_193910
Oracle DSN:           mpuiotdb_tp
Sample Rate:          100 Hz
Filtro labels:        ['LOW_ROT_ON', 'MEDIUM_ROT_ON', 'HIGH_ROT_O

In [3]:
# =============================================================================
# Celula 2: Carga do banco de dados
# =============================================================================
# --- Branch: dados ja filtrados pelo 01_EDA ---
LOADED_FROM_EDA = False
if globals().get('DATA_SOURCE') == 'EDA_FILTERED':
    import pandas as _pd2
    import numpy as _np2
    _csv_path = globals().get('CSV_FILE', '')
    if not os.path.exists(_csv_path):
        raise FileNotFoundError(f'CSV filtrado do 01_EDA nao encontrado: {_csv_path}')
    df_raw = _pd2.read_csv(_csv_path)
    RAW_SOURCE_PATH = _csv_path
    # Tipos
    if 'sample_rate' in df_raw.columns:
        df_raw['sample_rate'] = _pd2.to_numeric(df_raw['sample_rate'], errors='coerce')
    # Eixos derivados (caso nao estejam presentes)
    if 'vibration_dps' not in df_raw.columns and 'vibration' in df_raw.columns:
        df_raw['vibration_dps'] = _pd2.to_numeric(df_raw['vibration'], errors='coerce')
    if 'accel_mag_g' not in df_raw.columns and all(
            c in df_raw.columns for c in ('accel_x_g', 'accel_y_g', 'accel_z_g')):
        _ax = _pd2.to_numeric(df_raw['accel_x_g'], errors='coerce')
        _ay = _pd2.to_numeric(df_raw['accel_y_g'], errors='coerce')
        _az = _pd2.to_numeric(df_raw['accel_z_g'], errors='coerce')
        df_raw['accel_mag_g'] = _np2.sqrt(_ax**2 + _ay**2 + _az**2)
    # Sample rate do EDA meta (mais confiavel)
    if globals().get('EDA_META') and EDA_META.get('sample_hz'):
        EFFECTIVE_SAMPLE_RATE_HZ = float(EDA_META['sample_hz'])
    elif 'sample_rate' in df_raw.columns and df_raw['sample_rate'].notna().any():
        EFFECTIVE_SAMPLE_RATE_HZ = float(df_raw['sample_rate'].median())
    ACTIVE_CLASSES    = get_active_classes(df_raw)
    ACTIVE_COLOR_MAP  = get_color_map(df_raw)
    df = df_raw.copy()   # ja filtrado e balanceado pelo 01_EDA
    LOADED_FROM_EDA   = True
    print(f'[EDA_FILTERED] {len(df_raw):,} linhas de {_csv_path}')
    print(f'[EDA_FILTERED] EFFECTIVE_SAMPLE_RATE_HZ = {EFFECTIVE_SAMPLE_RATE_HZ}')
    print(f'[EDA_FILTERED] Classes: {ACTIVE_CLASSES}')
else:
    if 'DATA_SOURCE' not in globals():
        DATA_SOURCE = 'SQL'
    if 'CSV_FILE' not in globals():
        CSV_FILE = None

    RAW_SOURCE_PATH = None

    def _apply_filters_df(df):
        """Aplica filtros no DataFrame. Deriva composite label se possivel."""
        df_filtered = df.copy()
        df_filtered = derive_composite_label(df_filtered)

        if 'fan_state' in df_filtered.columns:
            if FILTER_LABELS:
                df_filtered = df_filtered[df_filtered['fan_state'].isin(FILTER_LABELS)].copy()
            else:
                df_filtered = df_filtered[df_filtered['fan_state'].isin(LEGACY_CLASS_ORDER)].copy()
        if FILTER_COLLECTION_ID and 'collection_id' in df_filtered.columns:
            if isinstance(FILTER_COLLECTION_ID, (list, tuple, set)):
                df_filtered = df_filtered[df_filtered['collection_id'].isin(list(FILTER_COLLECTION_ID))].copy()
            else:
                df_filtered = df_filtered[df_filtered['collection_id'] == FILTER_COLLECTION_ID].copy()
        if FILTER_SAMPLE_RATE_HZ is not None and 'sample_rate' in df_filtered.columns:
            df_filtered = df_filtered[df_filtered['sample_rate'] == FILTER_SAMPLE_RATE_HZ].copy()
        return df_filtered

    if DATA_SOURCE == 'CSV':
        data_dir = OUTPUT_DATA if 'OUTPUT_DATA' in globals() else os.path.join('output', 'data')
        if not CSV_FILE:
            raise ValueError('Fonte CSV selecionada, mas nenhum arquivo foi escolhido. Use a Celula 0.')
        csv_path = os.path.join(data_dir, CSV_FILE)
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f'CSV nao encontrado: {csv_path}')
        df_raw = pd.read_csv(csv_path)
        RAW_SOURCE_PATH = csv_path
        df_raw = _apply_filters_df(df_raw)
        print(f'CSV carregado: {csv_path}')
    else:
        # --- Carga via Oracle usando load_raw_from_oracle() de shared/data_sources.py ---
        _cid = FILTER_COLLECTION_ID if isinstance(FILTER_COLLECTION_ID, str) else None
        df_raw = load_raw_from_oracle(
            connection_str=get_engine(),
            collection_id=_cid,
        )
        RAW_SOURCE_PATH = f'Oracle:{ORACLE_DSN_STR}'

        # Derivar label composto e aplicar filtros em Python
        df_raw = _apply_filters_df(df_raw)
        print('Dados carregados do Oracle (com labels compostos).')

    if df_raw.empty:
        raise ValueError('Nenhum dado encontrado apos aplicar os filtros.')

    # Normalizar tipos
    if 'sample_rate' in df_raw.columns:
        df_raw['sample_rate'] = pd.to_numeric(df_raw['sample_rate'], errors='coerce')

    # Atualizar taxa efetiva (usa filtro ou taxa unica no banco)
    if 'sample_rate' in df_raw.columns and df_raw['sample_rate'].notna().any():
        unique_rates = sorted(df_raw['sample_rate'].dropna().unique().tolist())
        if FILTER_SAMPLE_RATE_HZ is not None:
            EFFECTIVE_SAMPLE_RATE_HZ = float(FILTER_SAMPLE_RATE_HZ)
        elif len(unique_rates) == 1:
            EFFECTIVE_SAMPLE_RATE_HZ = float(unique_rates[0])
        else:
            print(f'[AVISO] Multiplas sample_rate no dataset: {unique_rates}')
            print('[AVISO] Recomenda-se filtrar por sample_rate para consistencia.')
    else:
        print('[AVISO] Coluna sample_rate nao encontrada; usando control_state.')

    # --- Eixos derivados (para enriquecer pool de features) ---
    if 'vibration_dps' not in df_raw.columns and 'vibration' in df_raw.columns:
        df_raw['vibration_dps'] = pd.to_numeric(df_raw['vibration'], errors='coerce')

    if 'accel_mag_g' not in df_raw.columns and all(c in df_raw.columns for c in ('accel_x_g', 'accel_y_g', 'accel_z_g')):
        ax = pd.to_numeric(df_raw['accel_x_g'], errors='coerce')
        ay = pd.to_numeric(df_raw['accel_y_g'], errors='coerce')
        az = pd.to_numeric(df_raw['accel_z_g'], errors='coerce')
        df_raw['accel_mag_g'] = np.sqrt(ax**2 + ay**2 + az**2)

    # Determinar classes ativas e cores baseado nos dados carregados
    ACTIVE_CLASSES = get_active_classes(df_raw)
    ACTIVE_COLOR_MAP = get_color_map(df_raw)

    print(f'[INFO] Amostras carregadas: {len(df_raw)}')
    print(f'[INFO] EFFECTIVE_SAMPLE_RATE_HZ: {EFFECTIVE_SAMPLE_RATE_HZ} Hz')
    print(f'[INFO] Classes ativas: {ACTIVE_CLASSES}')
    print(f'[INFO] Distribuicao: {df_raw["fan_state"].value_counts().to_dict()}')

[EDA_FILTERED] 212,095 linhas de output\data\raw_sensor_data_filtered_20260307_193528.csv
[EDA_FILTERED] EFFECTIVE_SAMPLE_RATE_HZ = 100.0
[EDA_FILTERED] Classes: ['LOW_ROT_ON', 'MEDIUM_ROT_ON', 'HIGH_ROT_ON', 'LOW_ROT_OFF', 'MEDIUM_ROT_OFF', 'HIGH_ROT_OFF', 'FAN_OFF']


In [4]:
# =============================================================================
# Celula 3: Salvar dados brutos em CSV (backup do banco) com timestamp legivel
# =============================================================================
if globals().get('LOADED_FROM_EDA', False):
    print('[SKIP] Celula 3: dados ja exportados pelo 01_EDA.')
else:
    # Adicionar timestamp ISO 8601 legivel
    df_raw['timestamp_iso'] = pd.to_datetime(df_raw['timestamp'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')

    # Adicionar informacao de frequencia de amostragem
    df_raw['sample_rate_configured_hz'] = EFFECTIVE_SAMPLE_RATE_HZ

    # Calcular taxa real de amostragem por classe
    ts_col = df_raw['timestamp'].copy()
    if ts_col.median() > 1e12:
        ts_col = ts_col / 1000.0

    for cls in df_raw['fan_state'].unique():
        mask = df_raw['fan_state'] == cls
        t0 = ts_col[mask].min()
        duration = ts_col[mask].max() - t0
        n_samples = mask.sum()
        real_rate = n_samples / duration if duration > 0 else 0
        df_raw.loc[mask, 'sample_rate_real_hz'] = round(real_rate, 2)

    raw_csv_path = os.path.join(OUTPUT_DATA, f'raw_sensor_data_{TIMESTAMP_STR}.csv')
    df_raw.to_csv(raw_csv_path, index=False)
    print(f'Dados brutos salvos em: {raw_csv_path}')
    print(f'Shape: {df_raw.shape}')
    print(f'Colunas: {list(df_raw.columns)}')
    print(f'Exemplo timestamp_iso: {df_raw["timestamp_iso"].iloc[0]}')
    print(f'Taxa configurada: {EFFECTIVE_SAMPLE_RATE_HZ} Hz')
    print('Taxa real por classe:')
    for cls in ACTIVE_CLASSES:
        if cls in df_raw['fan_state'].unique():
            rate = df_raw.loc[df_raw['fan_state'] == cls, 'sample_rate_real_hz'].iloc[0]
            print(f'  {cls}: {rate} Hz')
        else:
            print(f'  {cls}: n/a')

[SKIP] Celula 3: dados ja exportados pelo 01_EDA.


In [5]:
# =============================================================================
# Celula 4: Normalizacao do tempo relativo por classe + taxa real
# =============================================================================
df_raw['timestamp_s'] = df_raw['timestamp'].copy()
# Se timestamp estiver em ms, converter para s
if df_raw['timestamp_s'].median() > 1e12:
    df_raw['timestamp_s'] = df_raw['timestamp_s'] / 1000.0

# Tempo relativo por classe (segundos desde o inicio da coleta daquela classe)
df_raw['relative_time_s'] = 0.0
for cls in df_raw['fan_state'].unique():
    mask = df_raw['fan_state'] == cls
    t0 = df_raw.loc[mask, 'timestamp_s'].min()
    df_raw.loc[mask, 'relative_time_s'] = df_raw.loc[mask, 'timestamp_s'] - t0

# Duracao e taxa real por classe
duration_per_class = df_raw.groupby('fan_state')['relative_time_s'].max()
samples_per_class = df_raw.groupby('fan_state').size()

print('=== Duracao por classe (s) ===')
print(duration_per_class)
print()
print(f'Amostras por classe: {samples_per_class.to_dict()}')
print()
print('=== Taxa real de amostragem (Hz) ===')
for cls in ACTIVE_CLASSES:
    if cls not in samples_per_class or cls not in duration_per_class or duration_per_class[cls] == 0:
        print(f'  {cls}: n/a')
        continue
    real_rate = samples_per_class[cls] / duration_per_class[cls]
    print(f'  {cls}: {real_rate:.2f} Hz (configurada: {EFFECTIVE_SAMPLE_RATE_HZ} Hz)')

# Consolidar metadata de validacao de taxa (com base nos dados carregados)
expected_hz = float(EFFECTIVE_SAMPLE_RATE_HZ) if 'EFFECTIVE_SAMPLE_RATE_HZ' in globals() else None
_sampling_stats = {}
for _cls in ACTIVE_CLASSES:
    if _cls not in samples_per_class or _cls not in duration_per_class or duration_per_class[_cls] == 0:
        continue
    _actual = float(samples_per_class[_cls] / duration_per_class[_cls])
    _dev = abs(_actual - expected_hz) / max(expected_hz, 1) * 100 if expected_hz is not None else None
    _sampling_stats[_cls] = {
        'samples': int(samples_per_class[_cls]),
        'duration_s': float(duration_per_class[_cls]),
        'actual_hz': round(_actual, 2),
        'deviation_pct': round(_dev, 2) if _dev is not None else None,
    }

VALIDATION_METADATA = {
    'status': 'computed_from_loaded_data',
    'expected_hz': expected_hz,
    'per_class': _sampling_stats,
    'note': f'Taxa estimada via timestamps das {len(ACTIVE_CLASSES)} classes ativas.',
}

=== Duracao por classe (s) ===
fan_state
FAN_OFF           299.988
HIGH_ROT_OFF      299.987
HIGH_ROT_ON       299.979
LOW_ROT_OFF       299.988
LOW_ROT_ON        299.988
MEDIUM_ROT_OFF    299.983
MEDIUM_ROT_ON     299.988
Name: relative_time_s, dtype: float64

Amostras por classe: {'FAN_OFF': 30303, 'HIGH_ROT_OFF': 30297, 'HIGH_ROT_ON': 30299, 'LOW_ROT_OFF': 30301, 'LOW_ROT_ON': 30298, 'MEDIUM_ROT_OFF': 30297, 'MEDIUM_ROT_ON': 30300}

=== Taxa real de amostragem (Hz) ===
  LOW_ROT_ON: 101.00 Hz (configurada: 100.0 Hz)
  MEDIUM_ROT_ON: 101.00 Hz (configurada: 100.0 Hz)
  HIGH_ROT_ON: 101.00 Hz (configurada: 100.0 Hz)
  LOW_ROT_OFF: 101.01 Hz (configurada: 100.0 Hz)
  MEDIUM_ROT_OFF: 101.00 Hz (configurada: 100.0 Hz)
  HIGH_ROT_OFF: 100.99 Hz (configurada: 100.0 Hz)
  FAN_OFF: 101.01 Hz (configurada: 100.0 Hz)


In [6]:
# =============================================================================
# Celula 5: Calculo da janela de tempo comum
# =============================================================================
common_window_s = duration_per_class.min()
print(f'Janela de tempo comum: {common_window_s:.2f} s')
print(f'Isso corresponde a ~{common_window_s/60:.1f} minutos')

Janela de tempo comum: 299.98 s
Isso corresponde a ~5.0 minutos


In [7]:
# =============================================================================
# Celula 6: FILTRO TEMPORAL - recortar TODAS as classes pela janela comum
# ANTES de qualquer analise posterior (graficos E features)
# =============================================================================
df = df_raw[df_raw['relative_time_s'] <= common_window_s].copy()

print(f'Amostras ANTES do filtro: {len(df_raw)}')
print(f'Amostras APOS o filtro:  {len(df)}')
print(f'Amostras por classe (filtrado): {df.groupby("fan_state").size().to_dict()}')
print(f'Duracao por classe (filtrado):')
print(df.groupby('fan_state')['relative_time_s'].max())

Amostras ANTES do filtro: 212095
Amostras APOS o filtro:  212089
Amostras por classe (filtrado): {'FAN_OFF': 30302, 'HIGH_ROT_OFF': 30296, 'HIGH_ROT_ON': 30299, 'LOW_ROT_OFF': 30300, 'LOW_ROT_ON': 30297, 'MEDIUM_ROT_OFF': 30296, 'MEDIUM_ROT_ON': 30299}
Duracao por classe (filtrado):
fan_state
FAN_OFF           299.978
HIGH_ROT_OFF      299.977
HIGH_ROT_ON       299.979
LOW_ROT_OFF       299.978
LOW_ROT_ON        299.978
MEDIUM_ROT_OFF    299.973
MEDIUM_ROT_ON     299.978
Name: relative_time_s, dtype: float64


In [8]:
# =============================================================================
# Celula 7: Engenharia de Features (janela deslizante - features estendidas)
#
# extract_features_windowed_extended calcula por janela:
#   - Basicas: std, range, rms  (alinhadas com JS, ddof=0)
#   - Shape:   skewness, kurtosis (excess)
#   - Percentis: P10, P25, P75, P90, P95
#   - FFT bands: fft_low (0-5Hz), fft_mid (5-15Hz), fft_high (15-10Hz)
#
# Total: 13 metricas x 8 eixos = 104 features candidatas
# =============================================================================

import importlib
import shared.feature_engineering as _fe_mod
importlib.reload(_fe_mod)
from shared.feature_engineering import extract_features_windowed_extended

# Aplicar sobre os dados FILTRADOS
all_features = []
for cls in ACTIVE_CLASSES:
    df_cls = df[df['fan_state'] == cls].reset_index(drop=True)
    if df_cls.empty:
        print(f'{cls}: 0 janelas (sem dados)')
        continue
    features = extract_features_windowed_extended(
        df_cls,
        cls,
        sensor_axes=FEATURE_AXES,
        window_size=WINDOW_SIZE,
        step_size=STEP_SIZE,
        timestamp_col='timestamp_s',
        sampling_hz=float(EFFECTIVE_SAMPLE_RATE_HZ),
    )
    all_features.extend(features)
    print(f'{cls}: {len(features)} janelas de {len(df_cls)} amostras')

df_features = pd.DataFrame(all_features)

# Colunas de features (numericas, excluindo metadados)
meta_cols = ['fan_state', 'collection_id', 'window_start', 'window_end', 'timestamp_start', 'timestamp_end', 'timestamp_mean']
feature_cols = [c for c in FEATURE_COLUMNS if c in df_features.columns]

print()
print(f'Total de janelas: {len(df_features)}')
print(f'Features por janela: {len(feature_cols)} (de {len(FEATURE_COLUMNS)} candidatas)')
print(f'Distribuicao: {df_features["fan_state"].value_counts().to_dict()}')

# =============================================================================
# Secao B — Features resistentes a deriva + momentos espectrais
# (complementam as features estendidas acima)
# =============================================================================
import importlib
import shared.feature_engineering as _fe_mod2
importlib.reload(_fe_mod2)
from shared.feature_engineering import (
    extract_features_windowed_drift_resistant,
    extract_features_windowed_spectral_moments,
)

_NFFT = 4096
_F_MIN = 0.5

dr_features = []
sm_features = []

for cls in ACTIVE_CLASSES:
    df_cls = df[df['fan_state'] == cls].reset_index(drop=True)
    if df_cls.empty:
        continue

    # Drift-resistant (shape + envelope features)
    dr = extract_features_windowed_drift_resistant(
        df_cls, cls,
        sensor_axes=SENSOR_AXES,
        window_size=WINDOW_SIZE,
        step_size=STEP_SIZE,
        timestamp_col='timestamp_s',
        sampling_hz=float(EFFECTIVE_SAMPLE_RATE_HZ),
        n_fft=_NFFT,
    )
    dr_features.extend(dr)

    # Momentos espectrais (P1-P14 por eixo)
    sm = extract_features_windowed_spectral_moments(
        df_cls, cls,
        sensor_axes=SENSOR_AXES,
        window_size=WINDOW_SIZE,
        step_size=STEP_SIZE,
        timestamp_col='timestamp_s',
        sampling_hz=float(EFFECTIVE_SAMPLE_RATE_HZ),
        n_fft=_NFFT,
        f_min=_F_MIN,
    )
    sm_features.extend(sm)
    print(f'  {cls}: {len(dr)} DR-janelas, {len(sm)} SM-janelas')

df_dr = pd.DataFrame(dr_features)
df_sm = pd.DataFrame(sm_features)

# Colunas de cada bloco
_meta_cols_set = {'fan_state', 'collection_id', 'window_start', 'window_end',
                  'timestamp_start', 'timestamp_end', 'timestamp_mean'}
dr_feat_cols = [c for c in df_dr.columns if c not in _meta_cols_set] if not df_dr.empty else []
sm_feat_cols = [c for c in df_sm.columns if c not in _meta_cols_set] if not df_sm.empty else []

print(f'\nDrift-resistant features: {len(dr_feat_cols)} colunas')
print(f'Spectral-moments features: {len(sm_feat_cols)} colunas')

# Merge: align on window_start + fan_state
if not df_dr.empty and not df_sm.empty:
    _merge_keys = ['fan_state', 'window_start']
    df_features_all = df_features.merge(
        df_dr[_merge_keys + dr_feat_cols], on=_merge_keys, how='left', suffixes=('', '_dr')
    ).merge(
        df_sm[_merge_keys + sm_feat_cols], on=_merge_keys, how='left', suffixes=('', '_sm')
    )
    all_feat_cols = (feature_cols +
                     [c for c in dr_feat_cols if c not in feature_cols] +
                     [c for c in sm_feat_cols if c not in feature_cols])
    feature_cols = [c for c in all_feat_cols if c in df_features_all.columns]
    df_features = df_features_all
    print(f'\nPool total de features: {len(feature_cols)} colunas')
    print(f'Total de janelas: {len(df_features)}')
else:
    print('[AVISO] DR ou SM features vazias — usando apenas features estendidas.')


LOW_ROT_ON: 1510 janelas de 30297 amostras
MEDIUM_ROT_ON: 1510 janelas de 30299 amostras
HIGH_ROT_ON: 1510 janelas de 30299 amostras
LOW_ROT_OFF: 1511 janelas de 30300 amostras
MEDIUM_ROT_OFF: 1510 janelas de 30296 amostras
HIGH_ROT_OFF: 1510 janelas de 30296 amostras
FAN_OFF: 1511 janelas de 30302 amostras

Total de janelas: 10572
Features por janela: 104 (de 104 candidatas)
Distribuicao: {'LOW_ROT_OFF': 1511, 'FAN_OFF': 1511, 'LOW_ROT_ON': 1510, 'MEDIUM_ROT_ON': 1510, 'HIGH_ROT_ON': 1510, 'MEDIUM_ROT_OFF': 1510, 'HIGH_ROT_OFF': 1510}
  LOW_ROT_ON: 1510 DR-janelas, 1510 SM-janelas
  MEDIUM_ROT_ON: 1510 DR-janelas, 1510 SM-janelas
  HIGH_ROT_ON: 1510 DR-janelas, 1510 SM-janelas
  LOW_ROT_OFF: 1511 DR-janelas, 1511 SM-janelas
  MEDIUM_ROT_OFF: 1510 DR-janelas, 1510 SM-janelas
  HIGH_ROT_OFF: 1510 DR-janelas, 1510 SM-janelas
  FAN_OFF: 1511 DR-janelas, 1511 SM-janelas

Drift-resistant features: 77 colunas
Spectral-moments features: 84 colunas

Pool total de features: 265 colunas
Total de

In [9]:
# =============================================================================
# Celula 8: Selecao de features - Cohen's d + correlacao por classe + pairwise + TOP-K
#           + Complemento ON vs OFF (garante cobertura dos pares ROT_ON/ROT_OFF)
# =============================================================================

import importlib
import shared.feature_selection as _fs_mod
importlib.reload(_fs_mod)
from shared.feature_selection import (
    select_features_anova_classwise_corr_pairwise_score_topk,
    select_features_cohens_d_classwise_corr_pairwise_score_topk,
)

# Parametros (ajuste fino aqui)
SELECTION_METHOD = 'cohens_d'  # 'cohens_d' (recomendado) ou 'anova'

MIN_COHENS_D        = 0.3   # d minimo global (todos os 21 pares)
COHENS_SCORE_MODE   = 'd_min_all'

CORRELATION_THRESHOLD   = 0.85
PAIRWISE_MIN_SEPARATION = 1.0
PAIRWISE_MIN_PAIRS      = 2
TOP_K_FEATURES          = min(16, len(feature_cols))
CORRELATION_MODE        = 'classwise_median'
SCORE_FORMULA           = COHENS_SCORE_MODE if SELECTION_METHOD == 'cohens_d' else 'min_sep * log(1 + F)'

# Complemento ON vs OFF
ON_OFF_PAIRS = [
    ('LOW_ROT_ON',    'LOW_ROT_OFF'),
    ('MEDIUM_ROT_ON', 'MEDIUM_ROT_OFF'),
    ('HIGH_ROT_ON',   'HIGH_ROT_OFF'),
]
ON_OFF_GUARANTEE_D = 1.5   # d minimo desejado para ao menos 1 feature por par ON/OFF
ON_OFF_MAX_ADD     = 4     # max de features adicionadas na fase de complemento

CLASSES = ACTIVE_CLASSES

# Aviso sobre classes com poucos dados
_class_window_counts = df_features['fan_state'].value_counts()
_small_classes = _class_window_counts[_class_window_counts < 5]
if not _small_classes.empty:
    print(f'[AVISO] Classes com poucas janelas (< 5):')
    for cls, cnt in _small_classes.items():
        print(f'  {cls}: {cnt} janelas')

# =============================================================================
# 1) Selecao principal (Cohen's d / ANOVA)
# =============================================================================
if SELECTION_METHOD == 'anova':
    ANOVA_ALPHA_SELECTION = ANOVA_ALPHA
    selected_features, df_scores, df_anova, df_significant = (
        select_features_anova_classwise_corr_pairwise_score_topk(
            df_features, feature_cols, class_col='fan_state', classes=CLASSES,
            anova_alpha=ANOVA_ALPHA_SELECTION,
            correlation_threshold=CORRELATION_THRESHOLD,
            correlation_mode=CORRELATION_MODE,
            pairwise_min_separation=PAIRWISE_MIN_SEPARATION,
            pairwise_min_pairs=PAIRWISE_MIN_PAIRS,
            top_k=TOP_K_FEATURES, verbose=True,
        )
    )
    _, _, df_sep = select_features_cohens_d_classwise_corr_pairwise_score_topk(
        df_features, feature_cols, class_col='fan_state', classes=CLASSES,
        correlation_threshold=1.0, correlation_mode=CORRELATION_MODE,
        pairwise_min_separation=0.0, pairwise_min_pairs=0,
        min_cohens_d=None, score_mode=COHENS_SCORE_MODE,
        top_k=len(feature_cols), verbose=False,
    )
else:
    selected_features, df_scores, df_sep = (
        select_features_cohens_d_classwise_corr_pairwise_score_topk(
            df_features, feature_cols, class_col='fan_state', classes=CLASSES,
            correlation_threshold=CORRELATION_THRESHOLD,
            correlation_mode=CORRELATION_MODE,
            pairwise_min_separation=PAIRWISE_MIN_SEPARATION,
            pairwise_min_pairs=PAIRWISE_MIN_PAIRS,
            min_cohens_d=MIN_COHENS_D,
            score_mode=COHENS_SCORE_MODE,
            top_k=TOP_K_FEATURES, verbose=True,
        )
    )
    _, _, df_anova, df_significant = (
        select_features_anova_classwise_corr_pairwise_score_topk(
            df_features, feature_cols, class_col='fan_state', classes=CLASSES,
            anova_alpha=ANOVA_ALPHA, correlation_threshold=1.0,
            correlation_mode=CORRELATION_MODE, pairwise_min_separation=0.0,
            pairwise_min_pairs=0, top_k=len(feature_cols), verbose=False,
        )
    )
    ANOVA_ALPHA_SELECTION = ANOVA_ALPHA

print(f'\n--- Selecao principal ---')
print(f'Features selecionadas: {len(selected_features)}')
if df_anova is not None and not df_anova.empty:
    print(f'ANOVA significativas: {len(df_significant)} de {len(df_anova)}')

# =============================================================================
# 2) Complemento ON vs OFF
#    Objetivo: garantir que ao menos 1 feature selecionada tenha
#    d >= ON_OFF_GUARANTEE_D para cada par ROT_ON / ROT_OFF.
#    Razao: a selecao por d_min_all (todos os 21 pares) pode sacrificar
#    features com excelente separacao ON/OFF mas d_min baixo em outros pares.
# =============================================================================

def _cohens_d_pair(feat, c1, c2, df=df_features):
    a = df[df['fan_state'] == c1][feat].dropna().values.astype(float)
    b = df[df['fan_state'] == c2][feat].dropna().values.astype(float)
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return 0.0
    sp = np.sqrt(((na-1)*np.var(a, ddof=1) + (nb-1)*np.var(b, ddof=1)) / (na+nb-2))
    return abs(a.mean() - b.mean()) / sp if sp > 1e-12 else 0.0

# Verifica cobertura atual (d_max entre features ja selecionadas, por par ON/OFF)
pairs_coverage = {}
for c1, c2 in ON_OFF_PAIRS:
    if c1 not in ACTIVE_CLASSES or c2 not in ACTIVE_CLASSES:
        continue
    ds = [_cohens_d_pair(f, c1, c2) for f in selected_features]
    pairs_coverage[(c1, c2)] = max(ds) if ds else 0.0

print(f'\n--- Cobertura ON vs OFF (selecao principal) ---')
for (c1, c2), d in pairs_coverage.items():
    status = 'OK' if d >= ON_OFF_GUARANTEE_D else 'INSUF'
    print(f'  [{status}] {c1} vs {c2}: d_max={d:.3f}  (threshold={ON_OFF_GUARANTEE_D})')

uncovered = {p: v for p, v in pairs_coverage.items() if v < ON_OFF_GUARANTEE_D}
added_features = []

if uncovered:
    print(f'\n[COMPLEMENTO] {len(uncovered)} par(es) com cobertura insuficiente.')
    candidates = [f for f in feature_cols if f not in selected_features]

    for (c1, c2) in sorted(uncovered, key=lambda p: pairs_coverage[p]):
        if len(added_features) >= ON_OFF_MAX_ADD:
            break
        # Melhor candidato nao ja adicionado
        cand_ds = [
            (f, _cohens_d_pair(f, c1, c2))
            for f in candidates
            if f not in added_features
        ]
        cand_ds.sort(key=lambda x: -x[1])
        for feat, d_val in cand_ds:
            if d_val >= ON_OFF_GUARANTEE_D:
                added_features.append(feat)
                print(f'  + {feat}  (d={d_val:.3f}, par={c1} vs {c2})')
                break
        else:
            # Nenhum candidato atinge o threshold — adiciona o melhor disponivel
            if cand_ds:
                feat, d_val = cand_ds[0]
                added_features.append(feat)
                print(f'  + {feat}  (d={d_val:.3f}, melhor disponivel para {c1} vs {c2})')
else:
    print('  Todos os pares ON vs OFF cobertos. Nenhuma feature adicional necessaria.')

selected_features = selected_features + added_features

# =============================================================================
# 3) Resultado final
# =============================================================================
print(f'\n--- Features finais: {len(selected_features)} ---')
print(f'  Base (Cohen\'s d / ANOVA): {len(selected_features) - len(added_features)}')
if added_features:
    print(f'  Complemento ON/OFF:       {len(added_features)} feature(s): {added_features}')
print()
for i, f in enumerate(selected_features, 1):
    # Busca d_min_all do df_sep para exibir
    row = df_sep[df_sep['feature'] == f]
    d_min = f'  d_min={float(row["d_min_all"].iloc[0]):.3f}' if not row.empty else ''
    tag = ' [ON/OFF compl.]' if f in added_features else ''
    print(f'  {i:02d}. {f}{d_min}{tag}')


Cohen's d ranking (top 15, 21 pares):
               feature  d_min_all
   gyro_x_dps_fft_high   1.313655
        gyro_x_dps_std   1.194498
      gyro_x_dps_sp_p2   1.085643
    accel_x_g_fft_high   0.949168
vibration_dps_fft_high   0.947902
     vibration_dps_std   0.904275
         accel_x_g_std   0.867220
       accel_z_g_sp_p2   0.788742
    accel_z_g_fft_high   0.785644
   vibration_dps_range   0.740957
         accel_z_g_std   0.718381
         accel_x_g_rms   0.550893
         accel_x_g_p10   0.524521
        gyro_x_dps_p25   0.515097
      accel_z_g_sp_p12   0.497820

Candidates with d_min_all >= 0.3: 45
Removidas por correlacao > 0.85: 16
Apos correlacao: 29

--- Filtro pairwise (min=1.0, pares>=2) ---
               feature    score  pairs_passed  d_min_all
   gyro_x_dps_fft_high 1.313655            21   1.313655
    accel_x_g_fft_high 0.949168            20   0.949168
vibration_dps_fft_high 0.947902            20   0.947902
       accel_z_g_sp_p2 0.788742            19   0.7

In [10]:
# =============================================================================
# Celula 9: Validacao robusta - Kruskal-Wallis + Mutual Information
#
# Kruskal-Wallis: teste nao-parametrico (nao assume normalidade)
# Mutual Information: captura relacoes nao-lineares entre feature e classe
# =============================================================================

# --- 1. Kruskal-Wallis (nao-parametrico) ---
kw_results = []
kw_skipped = []
for feat in feature_cols:
    groups = [df_features[df_features['fan_state'] == cls][feat].dropna().values
              for cls in ACTIVE_CLASSES]
    groups = [g for g in groups if len(g) > 1]  # remover classes sem dados
    if len(groups) < 2:
        continue
    # Verificar se ha variabilidade suficiente para o teste
    all_vals = np.concatenate(groups)
    if len(np.unique(all_vals)) <= 1:
        kw_skipped.append(feat)
        continue
    try:
        h_stat, p_val = kruskal(*groups)
        kw_results.append({'feature': feat, 'h_statistic': h_stat, 'p_value': p_val})
    except ValueError:
        kw_skipped.append(feat)

if kw_skipped:
    print(f'[INFO] {len(kw_skipped)} features puladas no KW (valores constantes): {kw_skipped}')

df_kw = pd.DataFrame(kw_results)
if not df_kw.empty:
    df_kw = df_kw.sort_values('h_statistic', ascending=False).reset_index(drop=True)
kw_significant = df_kw[df_kw['p_value'] < ANOVA_ALPHA]['feature'].tolist() if not df_kw.empty else []

print(f'=== Kruskal-Wallis ===')
print(f'Features significativas (p < {ANOVA_ALPHA}): {len(kw_significant)} de {len(df_kw)}')

# --- 2. Mutual Information ---
le = LabelEncoder()
y_encoded = le.fit_transform(df_features['fan_state'])
X_features = df_features[feature_cols]

mi_scores = mutual_info_classif(X_features, y_encoded, random_state=42)
df_mi = pd.DataFrame({'feature': feature_cols, 'mi_score': mi_scores}).sort_values('mi_score', ascending=False)

print(f'\n=== Mutual Information ===')
print(f'Top 15 features por MI:')
for _, row in df_mi.head(15).iterrows():
    print(f'  {row["feature"]:40s} MI={row["mi_score"]:.4f}')

# --- 3. Comparacao de rankings ---
df_ranking = pd.DataFrame({'feature': feature_cols})

# ANOVA rank (pode estar vazio se classes com dados insuficientes)
if 'df_anova' in globals() and df_anova is not None and not df_anova.empty:
    anova_rank = {row['feature']: i+1 for i, (_, row) in enumerate(df_anova.iterrows())}
    df_ranking['anova_rank'] = df_ranking['feature'].map(anova_rank)
else:
    df_ranking['anova_rank'] = float('nan')
    print('\n[AVISO] ANOVA nao produziu resultados — ranking ANOVA indisponivel.')

kw_rank = {row['feature']: i+1 for i, (_, row) in enumerate(df_kw.iterrows())} if not df_kw.empty else {}
df_ranking['kw_rank'] = df_ranking['feature'].map(kw_rank)
mi_rank = {row['feature']: i+1 for i, (_, row) in enumerate(df_mi.iterrows())}
df_ranking['mi_rank'] = df_ranking['feature'].map(mi_rank)
df_ranking['avg_rank'] = df_ranking[['anova_rank', 'kw_rank', 'mi_rank']].mean(axis=1)
df_ranking = df_ranking.sort_values('avg_rank')

print(f'\n=== Ranking Consensual (media dos metodos disponiveis) ===')
print(f'Top 20 features:')
for _, row in df_ranking.head(20).iterrows():
    in_selected = '*' if row['feature'] in selected_features else ' '
    anova_str = f'{row["anova_rank"]:3.0f}' if pd.notna(row.get('anova_rank')) else '  -'
    kw_str = f'{row["kw_rank"]:3.0f}' if pd.notna(row.get('kw_rank')) else '  -'
    print(f'  {in_selected} {row["feature"]:40s} ANOVA={anova_str}  KW={kw_str}  MI={row["mi_rank"]:3.0f}  AVG={row["avg_rank"]:.1f}')
print(f'\n  * = presente nas features selecionadas (Top-K)')

# Verificar concordancia
if 'df_significant' in globals() and df_significant is not None and not df_significant.empty:
    anova_set = set(df_significant['feature'].tolist())
else:
    anova_set = set()
kw_set = set(kw_significant)
print(f'\nConcordancia ANOVA vs Kruskal-Wallis:')
print(f'  Ambos significativos: {len(anova_set & kw_set)}')
print(f'  So ANOVA: {len(anova_set - kw_set)}')
print(f'  So KW: {len(kw_set - anova_set)}')

=== Kruskal-Wallis ===
Features significativas (p < 0.05): 265 de 265

=== Mutual Information ===
Top 15 features por MI:
  gyro_z_dps_top2_freq_ratio               MI=1.6951
  accel_x_g_top2_freq_ratio                MI=1.6430
  accel_mag_g_range                        MI=1.5927
  accel_x_g_range                          MI=1.5829
  accel_x_g_fft_high                       MI=1.5828
  accel_x_g_std                            MI=1.5737
  gyro_y_dps_top2_freq_ratio               MI=1.5459
  gyro_x_dps_top2_freq_ratio               MI=1.5456
  accel_mag_g_p95                          MI=1.5143
  accel_x_g_p95                            MI=1.5137
  vibration_dps_range                      MI=1.5136
  accel_y_g_top2_freq_ratio                MI=1.5097
  accel_mag_g_p90                          MI=1.5077
  accel_mag_g_std                          MI=1.4935
  gyro_x_dps_range                         MI=1.4861

=== Ranking Consensual (media dos metodos disponiveis) ===
Top 20 features:
  * ac

In [11]:
# =============================================================================
# Celula 10: Exportar artefatos
# =============================================================================

from itertools import combinations as _combinations
from shared.traceability import hash_file, save_json, append_registry

# Sampling validation metadata (opcional)
if 'VALIDATION_METADATA' not in globals():
    VALIDATION_METADATA = {
        'status': 'not_available',
        'expected_hz': float(EFFECTIVE_SAMPLE_RATE_HZ) if 'EFFECTIVE_SAMPLE_RATE_HZ' in globals() else None,
        'per_class': {},
        'reason': 'VALIDATION_METADATA nao foi calculada antes da exportacao.',
    }

# Extrair collection_ids usados nos dados
collection_ids_used = df_raw['collection_id'].unique().tolist() if 'collection_id' in df_raw.columns else []

# Garantir collection_id no dataframe de features (para GroupKFold e rastreabilidade)
if 'collection_id' in df_raw.columns and 'collection_id' not in df_features.columns:
    # Mapear collection_id mais frequente por classe
    col_id_per_class = df_raw.groupby('fan_state')['collection_id'].agg(lambda x: x.mode().iloc[0] if len(x) > 0 else 'unknown').to_dict()
    df_features['collection_id'] = df_features['fan_state'].map(col_id_per_class)

# 1. CSV de features extraidas (com timestamp e collection_id)
features_csv_path = os.path.join(OUTPUT_DATA, f'features_extracted_{TIMESTAMP_STR}.csv')
df_features.to_csv(features_csv_path, index=False)
print(f'Features CSV salvo: {features_csv_path}')

# Copia "latest" para o notebook 03
latest_features_path = os.path.join(OUTPUT_DATA, 'features_latest.csv')
df_features.to_csv(latest_features_path, index=False)
print(f'Features latest CSV: {latest_features_path}')

# Hash do dataset de features
features_csv_hash = hash_file(features_csv_path)

# 2. Summary JSON
summary = {
    'timestamp': TIMESTAMP_STR,
    'raw_samples': int(len(df_raw)),
    'filtered_samples': int(len(df)),
    'common_window_s': float(common_window_s),
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE,
    'total_windows': int(len(df_features)),
    'feature_count': len(feature_cols),
    'selected_feature_count': len(selected_features),
    'class_distribution_raw': {k: int(v) for k, v in df_raw['fan_state'].value_counts().to_dict().items()},
    'class_distribution_filtered': {k: int(v) for k, v in df['fan_state'].value_counts().to_dict().items()},
    'class_distribution_windows': {k: int(v) for k, v in df_features['fan_state'].value_counts().to_dict().items()},
    'anova_alpha': ANOVA_ALPHA,
    'correlation_threshold': CORRELATION_THRESHOLD,
    'correlation_mode': CORRELATION_MODE,
    'pairwise_min_separation': PAIRWISE_MIN_SEPARATION,
    'pairwise_min_pairs': PAIRWISE_MIN_PAIRS,
    'top_k': TOP_K_FEATURES,
    'score_formula': SCORE_FORMULA,
    'selection_method': (SELECTION_METHOD if 'SELECTION_METHOD' in globals() else None),
    'min_cohens_d': (MIN_COHENS_D if 'MIN_COHENS_D' in globals() else None),
    'cohens_score_mode': (COHENS_SCORE_MODE if 'COHENS_SCORE_MODE' in globals() else None),
    'selected_features': selected_features,
    'sampling_validation': (VALIDATION_METADATA if 'VALIDATION_METADATA' in globals() else None),
    'collection_ids': collection_ids_used,
    'features_csv_hash': features_csv_hash,
    'n_classes': len(ACTIVE_CLASSES),
    'classes': ACTIVE_CLASSES,
}

summary_path = os.path.join(OUTPUT_METRICS, 'analise_exploratoria_summary.json')
save_json(summary_path, summary)
print(f'Summary salvo: {summary_path}')

# 3. Feature config JSON (para frontend e notebook 03)
# --- Versionamento automatico: incrementa minor version a cada execucao ---
config_path = os.path.join(CONFIG_DIR, 'feature_config.json')
previous_version = '0.0'
previous_history = []
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        old_config = json.load(f)
    previous_version = old_config.get('version', '0.0')
    previous_history = old_config.get('version_history', [])

# Incrementar versao: major.minor -> major.(minor+1)
try:
    parts = previous_version.split('.')
    new_version = f'{parts[0]}.{int(parts[1]) + 1}' if len(parts) >= 2 else f'{int(parts[0]) + 1}.0'
except (ValueError, IndexError):
    new_version = '1.0'

# IDs de rastreabilidade
# Usa EDA_ID do 01_EDA (via eda_latest_meta.json) se disponivel, senao gera novo
if not globals().get('EDA_ID'):
    EDA_ID = f'eda_{TIMESTAMP_STR}'
FE_ID = f'fe_{TIMESTAMP_STR}'

# Metodo de selecao (para rastreabilidade)
if 'SELECTION_METHOD' not in globals():
    SELECTION_METHOD = 'anova'
METHOD_ID = (
    'cohens_d_min_adjacent_classwise_corr_pairwise_score_topk'
    if SELECTION_METHOD == 'cohens_d'
    else 'anova_f_test_with_classwise_correlation_pairwise_2of3_score_topk'
)

# Construir cohens_d_ranking dinamicamente (todos os pares presentes no df_sep)
_d_pair_cols = [c for c in df_sep.columns if c.startswith('d_') and c not in ('d_min_adjacent', 'd_min_all')]
_cohens_ranking = []
for _, row in df_sep.iterrows():
    if row.get('feature') not in selected_features:
        continue
    entry = {
        'feature': row['feature'],
        'd_min_all': float(row.get('d_min_all', 0.0)),
    }
    for col in _d_pair_cols:
        entry[col] = float(row.get(col, 0.0))
    _cohens_ranking.append(entry)

feature_config = {
    'version': new_version,
    'eda_id': EDA_ID,
    'fe_id': FE_ID,
    'eda_source': globals().get('EDA_META', {}) or {},  # meta do 01_EDA
    'generated_by': '02_Feature_Engineering.ipynb',
    'generated_at': TIMESTAMP_STR,
    'sample_rate_hz': EFFECTIVE_SAMPLE_RATE_HZ,
    'sampling_validation': (VALIDATION_METADATA if 'VALIDATION_METADATA' in globals() else None),
    'collection_ids': collection_ids_used,
    'features_csv_hash': features_csv_hash,
    'selection_criteria': {
        'anova_alpha': ANOVA_ALPHA,
        'correlation_threshold': CORRELATION_THRESHOLD,
        'correlation_mode': CORRELATION_MODE,
        'pairwise_min_separation': PAIRWISE_MIN_SEPARATION,
        'pairwise_min_pairs': PAIRWISE_MIN_PAIRS,
        'score_formula': SCORE_FORMULA,
        'top_k': TOP_K_FEATURES,
        'method': METHOD_ID,
        'selection_method': SELECTION_METHOD,
        'min_cohens_d': (MIN_COHENS_D if 'MIN_COHENS_D' in globals() else None),
        'cohens_score_mode': (COHENS_SCORE_MODE if 'COHENS_SCORE_MODE' in globals() else None),
    },
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE,
    'selected_features': selected_features,
    'all_features': feature_cols,
    'feature_count': len(selected_features),
    'total_feature_count': len(feature_cols),
    'n_classes': len(ACTIVE_CLASSES),
    'classes': ACTIVE_CLASSES,
    'class_distribution': {k: int(v) for k, v in df_features['fan_state'].value_counts().to_dict().items()},
    'sensor_axes': SENSOR_AXES,
    'derived_axes': (DERIVED_AXES if 'DERIVED_AXES' in globals() else []),
    'feature_axes': FEATURE_AXES,
    'feature_metrics': FEATURE_METRICS,
    'cohens_d_ranking': _cohens_ranking,
    'anova_ranking': [
        {
            'feature': row['feature'],
            'f_statistic': float(row['f_statistic']),
            'p_value': float(row['p_value']),
        }
        for _, row in (df_significant.iterrows() if 'df_significant' in globals() and df_significant is not None and not df_significant.empty else iter([]))
        if row['feature'] in selected_features
    ],
    'version_history': previous_history + [
        {
            'version': new_version,
            'eda_id': EDA_ID,
            'fe_id': FE_ID,
            'date': TIMESTAMP_STR,
            'features_count': len(selected_features),
            'n_classes': len(ACTIVE_CLASSES),
            'sample_rate_hz': EFFECTIVE_SAMPLE_RATE_HZ,
            'method': METHOD_ID,
        }
    ],
}

save_json(config_path, feature_config)

print()
print(f'Feature config salvo: {config_path}')
print(f'  Versao: {new_version} (anterior: {previous_version})')
print(f'  EDA ID: {EDA_ID}')
print(f'  FE ID:  {FE_ID}')
print(f'  Features: {len(selected_features)}')
print(f'  Classes: {len(ACTIVE_CLASSES)} ({", ".join(ACTIVE_CLASSES)})')
print(f'  Sample rate: {EFFECTIVE_SAMPLE_RATE_HZ} Hz')
print(f'  Sampling validation: {VALIDATION_METADATA.get("status", "N/A")}')
print(f'  Pairwise separation min (>= 2/3 pares): {PAIRWISE_MIN_SEPARATION}')
print(f'  Collection IDs: {collection_ids_used}')

# 4. Feature engineering run config
fe_run = {
    'generated_at': TIMESTAMP_STR,
    'eda_id': EDA_ID,
    'fe_id': FE_ID,
    'eda_source': globals().get('EDA_META', {}) or {},  # meta do 01_EDA
    'feature_config_version': new_version,
    'raw_csv_path': raw_csv_path if 'raw_csv_path' in globals() else None,
    'features_csv_path': features_csv_path,
    'features_latest_path': latest_features_path,
    'features_csv_hash': features_csv_hash,
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE,
    'sample_rate_hz': EFFECTIVE_SAMPLE_RATE_HZ,
    'selected_features': selected_features,
    'selection_criteria': feature_config['selection_criteria'],
}

fe_run_path = os.path.join(OUTPUT_METRICS, 'feature_engineering_run.json')
save_json(fe_run_path, fe_run)
print(f'FE run config salvo: {fe_run_path}')

# --- Salvar configuracao resumida da execucao para notebook 03 (compat) ---
eda_run_config = {
    'generated_at': TIMESTAMP_STR,
    'eda_id': EDA_ID,
    'fe_id': FE_ID,
    'eda_source': globals().get('EDA_META', {}) or {},  # meta do 01_EDA
    'feature_config_version': new_version,
    'raw_csv_path': raw_csv_path if 'raw_csv_path' in globals() else None,
    'features_csv_path': features_csv_path,
    'features_latest_path': latest_features_path,
    'features_csv_hash': features_csv_hash,
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE,
    'sample_rate_hz': EFFECTIVE_SAMPLE_RATE_HZ,
    'selected_features': selected_features,
}

eda_config_path = os.path.join(OUTPUT_METRICS, 'eda_run_config.json')
save_json(eda_config_path, eda_run_config)
print(f'EDA run config salvo: {eda_config_path}')

# 5. Pipeline registry
registry_path = os.path.join(OUTPUT_METRICS, 'pipeline_registry.json')
append_registry(registry_path, {
    'type': 'feature_engineering',
    'timestamp': TIMESTAMP_STR,
    'eda_id': EDA_ID,
    'fe_id': FE_ID,
    'eda_source': globals().get('EDA_META', {}) or {},  # meta do 01_EDA
    'feature_config_version': new_version,
    'features_csv_path': features_csv_path,
    'features_csv_hash': features_csv_hash,
    'selected_features_count': len(selected_features),
    'n_classes': len(ACTIVE_CLASSES),
    'selection_method': feature_config['selection_criteria']['method'],
    'collection_ids': collection_ids_used,
})

print()
print('=== CONCLUIDO ===')
print('Proximo passo: executar 03_Model_Training_Evaluation.ipynb')
print(f'O notebook 03 usara: {latest_features_path}')
print(f'O modelo gerado tera rastreabilidade para EDA ID: {EDA_ID}')

Features CSV salvo: c:\Users\Anderson\Downloads\oracle_fast_api_iot_esp32_MPU6050_project\notebooks\output\data\features_extracted_20260307_193910.csv
Features latest CSV: c:\Users\Anderson\Downloads\oracle_fast_api_iot_esp32_MPU6050_project\notebooks\output\data\features_latest.csv
Summary salvo: c:\Users\Anderson\Downloads\oracle_fast_api_iot_esp32_MPU6050_project\notebooks\output\metrics\analise_exploratoria_summary.json

Feature config salvo: c:\Users\Anderson\Downloads\oracle_fast_api_iot_esp32_MPU6050_project\config\feature_config.json
  Versao: 5.19 (anterior: 5.18)
  EDA ID: eda_20260307_192958
  FE ID:  fe_20260307_193910
  Features: 16
  Classes: 7 (LOW_ROT_ON, MEDIUM_ROT_ON, HIGH_ROT_ON, LOW_ROT_OFF, MEDIUM_ROT_OFF, HIGH_ROT_OFF, FAN_OFF)
  Sample rate: 100.0 Hz
  Sampling validation: computed_from_loaded_data
  Pairwise separation min (>= 2/3 pares): 1.0
  Collection IDs: ['col_20260307_185012_100hz']
FE run config salvo: c:\Users\Anderson\Downloads\oracle_fast_api_iot_esp3